# Praxis · Reproduce our validation — *The Volatility Edge* (VIX ETNs, #372649)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/finique/praxis-reproductions/blob/main/372649-the-volatility-edge.ipynb)

This notebook **reproduces our exact verdict** on the paper, from **public data only** —
no account, no paid feeds, nothing to trust on faith. It:
1. rebuilds the **S&P 500 VIX Short-Term Futures Index** from free **CBOE VIX-futures settlements**
   (the paper used paid Norgate; the live ETNs are survivorship-broken — XIV died, SVXY re-levered, VXX split 8×),
2. runs the paper's **four rule sets** (eVRP + VIX term-structure, sized at VIX/100),
3. prints **claimed → measured** so you can check the alpha yourself.

**Our published verdict: VALIDATED — but ~0.9, not the 1.0 headline.** Full report:
https://market-graphs.com/praxis/papers/372649

> **Same public data, same logic as our engine — this lands on our exact published numbers.** It runs the headline reproduction (claimed → measured across the 4 rule sets); the full **C0–C7 gate suite** (deflation, crisis, decay, originality) lives in the report. Runs in ~1–2 min (it fetches ~230 CBOE contracts).


In [ ]:
!pip install -q "yfinance==0.2.65" requests


In [ ]:
"""Praxis · reproduce our validation of paper 372649 "The Volatility Edge" (VIX ETNs).
Self-contained, PUBLIC data only (CBOE + Yahoo). Reproduces claimed→measured Sharpe.
This is the notebook logic, tested as a plain script first."""
from __future__ import annotations
import calendar, datetime as dt, io, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, requests
warnings.filterwarnings("ignore")

UA = {"User-Agent": "Mozilla/5.0 (praxis-repro)"}
CODE = {1:"F",2:"G",3:"H",4:"J",5:"K",6:"M",7:"N",8:"Q",9:"U",10:"V",11:"X",12:"Z"}
START, END = "2008-01-01", "2025-06-01"

# ---------- public index signals: VIX + VIX3M from CBOE, SPY from Yahoo ----------
def yf_close(ticker, adjusted=False, tries=3):
    """Yahoo daily close, with retry (yfinance can 429/return partial from cloud IPs)."""
    import time, yfinance as yf
    for i in range(tries):
        s = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=adjusted)
        if s is not None and not s.empty and len(s) > 100:
            c = s["Close"]; return (c.iloc[:, 0] if isinstance(c, pd.DataFrame) else c).rename(ticker)
        time.sleep(2 * (i + 1))
    raise RuntimeError(f"yfinance failed for {ticker}")

# ---------- reconstruct the S&P 500 VIX Short-Term Futures Index from CBOE VX ----------
def _third_friday(y, m):
    return [dt.date(y, m, d) for d in range(1, calendar.monthrange(y, m)[1]+1)
            if dt.date(y, m, d).weekday() == 4][2]

def _nominal(y, m):
    ny, nm = (y+1, 1) if m == 12 else (y, m+1)
    return _third_friday(ny, nm) - dt.timedelta(days=30)

def _get(u):
    try:
        r = requests.get(u, headers=UA, timeout=30)
        return r.text if r.status_code == 200 and "Settle" in r.text else None
    except Exception:
        return None

def _parse(txt):
    df = pd.read_csv(io.StringIO(txt), on_bad_lines="skip")
    df.columns = [c.strip() for c in df.columns]
    if "Trade Date" not in df.columns or "Settle" not in df.columns:
        return pd.Series(dtype=float)
    d = pd.to_datetime(df["Trade Date"], errors="coerce")
    s = pd.to_numeric(df["Settle"], errors="coerce")
    m = d.notna() & s.notna() & (s > 0)
    out = pd.Series(s[m].to_numpy(), index=pd.DatetimeIndex(d[m])).sort_index()
    return out[~out.index.duplicated(keep="last")]

def _contract(y, m):
    nom = _nominal(y, m)
    if nom >= dt.date(2014, 1, 1):                          # modern per-contract CSV
        for dd in (0, -1, -2, 1):
            t = _get(f"https://cdn.cboe.com/data/us/futures/market_statistics/historical_data/"
                     f"VX/VX_{(nom+dt.timedelta(days=dd)).isoformat()}.csv")
            if t: return _parse(t)
    else:                                                   # legacy archive (full history ≤2013)
        t = _get(f"https://cdn.cboe.com/resources/futures/archive/volume-and-price/"
                 f"CFE_{CODE[m]}{y%100:02d}_VX.csv")
        if t: return _parse(t)
    return pd.Series(dtype=float)

def reconstruct():
    jobs = [(y, m) for y in range(2007, 2026) for m in range(1, 13)]
    with ThreadPoolExecutor(16) as ex:
        res = list(ex.map(lambda ym: _contract(*ym), jobs))
    cols = {s.index.max().normalize(): s for s in res if len(s) > 5}
    panel = pd.DataFrame(cols).sort_index().reindex(sorted(cols), axis=1)
    exp = list(panel.columns); arr = np.array([e.date() for e in exp])
    recs = []
    for t in panel.index:
        fut = np.where(arr > t.date())[0]
        if len(fut) < 2: continue
        i = fut[0]; f, n = exp[i], exp[i+1]
        prev = exp[i-1].date() if i > 0 else (f.date()-dt.timedelta(days=30))
        dtt = np.busday_count(prev, f.date()); drr = np.busday_count(t.date(), f.date())
        if dtt <= 0: continue
        wf = min(max(drr/dtt, 0.0), 1.0); recs.append((t, f, n, wf, 1-wf))
    pos = pd.DataFrame(recs, columns=["d","f","n","wf","wn"]).set_index("d")
    out, prev = {}, None
    for t, r in pos.iterrows():
        if prev is not None:
            t0, f0, n0, wf0, wn0 = prev
            try:
                v1 = wf0*panel.at[t, f0] + wn0*panel.at[t, n0]
                v0 = wf0*panel.at[t0, f0] + wn0*panel.at[t0, n0]
                if v0 == v0 and v1 == v1 and v0 != 0: out[t] = v1/v0 - 1
            except KeyError: pass
        prev = (t, r["f"], r["n"], r["wf"], r["wn"])
    return pd.Series(out).sort_index().loc[START:END].rename("idx_ret")

# ---------- signals + the 4 rule sets (paper Table 2) ----------
def target(strategy, evrp, vix, contango):
    n = len(evrp); leg = np.zeros(n, int); size = np.zeros(n)
    back = ~contango
    if strategy == 1: leg[:] = -1; size[:] = .20; return leg, size
    if strategy == 2:
        m = evrp > 0; leg[m] = -1; size[m] = .20; return leg, size
    sf = (evrp > 0) & contango; sh = (evrp <= 0) & contango; lo = (evrp <= 0) & back
    if strategy == 3:
        leg[sf]=-1; size[sf]=.20; leg[sh]=-1; size[sh]=.10; leg[lo]=1; size[lo]=.20; return leg, size
    v = vix/100.0
    leg[sf]=-1; size[sf]=v[sf]; leg[sh]=-1; size[sh]=.5*v[sh]; leg[lo]=1; size[lo]=v[lo]; return leg, size

def backtest(idx_ret, vix, vix3m, spy, strategy, fee_bps=5.0, band=.02, win=10, etn_fee=.0095):
    df = pd.concat([idx_ret.rename("r"), vix.rename("vix"), vix3m.rename("vix3m"),
                    spy.rename("spy")], axis=1).dropna()             # align all calendars first
    rv = np.log(df["spy"]/df["spy"].shift(1)).rolling(win).std(ddof=1)*np.sqrt(252)*100
    df["evrp"] = df["vix"] - rv
    df["contango"] = df["vix"] < df["vix3m"]
    df = df.dropna()
    leg, size = target(strategy, df["evrp"].to_numpy(), df["vix"].to_numpy(), df["contango"].to_numpy())
    r = df["r"].to_numpy(); n = len(df); net = np.zeros(n); held_leg, held_f = 0, 0.0; trades = 0
    for t in range(n-1):
        tl, tf = int(leg[t]), float(size[t]); tf = 0.0 if tl == 0 else tf
        flip = (tl != held_leg) and not (tf == 0 and held_f == 0)
        turn = held_f+tf if flip else (abs(tf-held_f) if abs(tf-held_f) > band else 0.0)
        if turn > 0: held_leg = tl if tf > 0 else 0; held_f = tf; trades += 1
        lr = held_leg*r[t+1] if held_leg != 0 else 0.0
        pr = held_f*lr
        net[t+1] = pr - turn*(fee_bps/1e4) - held_f*(etn_fee/252)
        if held_f > 0 and held_leg != 0 and (1+pr) != 0: held_f = held_f*(1+lr)/(1+pr)
    return pd.Series(net, index=df.index).iloc[1:]

def sharpe(x, ppy): x = x.dropna(); return x.mean()/x.std(ddof=1)*np.sqrt(ppy)
def cagr(x): x = x.dropna(); return (1+x).prod()**(252/len(x))-1
def mdd(x): w = (1+x.dropna()).cumprod(); return (w/w.cummax()-1).min()


In [ ]:
print("Fetching public data (Yahoo VIX/VIX3M/SPY) + reconstructing the index from CBOE VIX futures…")
vix, vix3m, spy = yf_close("^VIX"), yf_close("^VIX3M"), yf_close("SPY", adjusted=True)
idx = reconstruct()
print(f"reconstructed index: {len(idx)} days {idx.index.min().date()} → {idx.index.max().date()}\n")

claim = {1:.48, 2:.56, 3:.87, 4:1.00}
name  = {1:"1 Passive", 2:"2 eVRP", 3:"3 eVRP+BoC", 4:"4 eVRP+BoC+Sizing"}
print(f"{'Strategy':20}{'claimed SR':>11}{'daily SR':>10}{'monthly SR':>12}{'CAGR%':>8}{'MDD%':>7}")
rets = {}
for s in (1,2,3,4):
    r = backtest(idx, vix, vix3m, spy, s); rets[s] = r
    mo = (1+r).resample("ME").prod()-1
    print(f"{name[s]:20}{claim[s]:>11}{sharpe(r,252):>10.2f}{sharpe(mo,12):>12.2f}{cagr(r)*100:>8.1f}{mdd(r)*100:>7.1f}")

# Strategy 4 — growth of $1 (log) + drawdown
import matplotlib.pyplot as plt
eq = (1+rets[4]).cumprod(); dd = eq/eq.cummax()-1
fig, ax = plt.subplots(2,1, figsize=(10,6), sharex=True, gridspec_kw={"height_ratios":[2,1]})
eq.plot(ax=ax[0], color="#6D1F2C", logy=True); ax[0].set_title("Strategy 4 — growth of $1 (log)"); ax[0].set_ylabel("×")
(dd*100).plot(ax=ax[1], color="#6D1F2C"); ax[1].fill_between(dd.index, dd*100, 0, color="#6D1F2C", alpha=.12)
ax[1].set_title("Drawdown"); ax[1].set_ylabel("%"); plt.tight_layout(); plt.show()


## What you just reproduced
- **Strategy 4 (dual-signal + sizing): daily Sharpe ~1.0, tail-honest monthly ~0.9, CAGR ~17%, max drawdown −31%** — matching our report.
- The **passive short-vol book** (0.48) is mediocre; the **dual signal** (term-structure → cash/long-vol) is the value-add — it sidestepped Volmageddon-2018 (flat into the +97% spike that killed XIV).

## The honest caveats (why it's VALIDATED, not celebrated)
- The **1.0 daily Sharpe leans on 2008–2009** (the post-GFC vol collapse, the strategy's best year); the robust number is the **monthly ~0.9**.
- The **dynamic sizing lifts CAGR but not risk-adjusted return** (fixed-size S3 ≥ sized S4, risk-adjusted).
- Real max drawdown is **−31%** (COVID-2020), not the paper's smoothed *'−12%'* headline.
- It's a **2025 paper — no true post-publication out-of-sample** yet.

**This is Praxis: not a strategy to trade — a reproducible audit of whether the alpha is real.**
